In [1]:
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import HDBSCAN
from stopwordsiso import stopwords
from tqdm import tqdm
import ast
from tqdm.contrib.concurrent import process_map

import warnings
warnings.simplefilter("ignore")

/hpc/home/as1676/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../../../data/comp_ideology_detection/interventions_us_113_with_embeddings.csv', index_col=0)

In [3]:
df['word_count'] = df['speech'].str.split().str.len()
df = df[df['word_count'] <= 600]
print(f"nrows: {df.shape[0]}")

tqdm.pandas()
df['embedding'] = df['embedding'].progress_apply(ast.literal_eval)

docs = df['speech'].tolist()
embeddings = np.vstack(df['embedding'].values)

nrows: 67802


100%|██████████| 67802/67802 [02:27<00:00, 459.48it/s]


In [7]:
custom_stopwords = [
    'mr', 'mrs', 'madam', 'speaker', 'president', 'gentleman', 'gentlewoman',
    'yield', 'time', 'today', 'would', 'like', 'also', 'one', 'us', 'say',
    'want', 'know', 'think', 'thank', 'colleague', 'member', 'congress',
    'house', 'senate', 'legislation', 'bill', 'act', 'ask', 'unanimous',
    'consent', 'rise'
]
all_stopwords = list(set(list(stopwords('en')) + custom_stopwords))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    min_df=10,          
    max_df=0.5,       
    ngram_range=(1, 2)
)

In [14]:
umap_model = UMAP(random_state=123, n_jobs=12)
hdbscan_model = HDBSCAN(min_cluster_size=100, n_jobs=12)

topic_model = BERTopic(verbose=True,
                      umap_model=umap_model,
                      hdbscan_model=hdbscan_model,
                      vectorizer_model=CountVectorizer(stop_words=list(stopwords('en'))))

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-02-12 16:46:07,909 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-12 16:47:17,367 - BERTopic - Dimensionality - Completed ✓
2026-02-12 16:47:17,369 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-12 16:47:40,764 - BERTopic - Cluster - Completed ✓
2026-02-12 16:47:40,775 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-12 16:47:44,429 - BERTopic - Representation - Completed ✓


In [15]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,28501,-1_time_speaker_president_people,"[time, speaker, president, people, amendment, ...",[I thank the gentleman from California. Chairm...
1,0,1577,0_care_health_insurance_obamacare,"[care, health, insurance, obamacare, affordabl...",[Mr. Speaker. as we see ObamaCare go into effe...
2,1,1525,1_energy_oil_gas_pipeline,"[energy, oil, gas, pipeline, coal, climate, ke...",[Madam President. obviously what is happening ...
3,2,1346,2_budget_debt_spending_tax,"[budget, debt, spending, tax, cuts, deficit, b...",[I thank the gentleman for yielding. Mr. Chair...
4,3,950,3_minutes_yield_gentleman_minute,"[minutes, yield, gentleman, minute, illinois, ...",[Mr. Speaker. I yield 2 minutes to the gentlem...
...,...,...,...,...,...
142,141,103,141_science_epa_data_nuclear,"[science, epa, data, nuclear, scientific, advi...",[Mr. Chairman. I oppose this bill. I really be...
143,142,102,142_absence_quorum_withhold_requisite,"[absence, quorum, withhold, requisite, suggest...","[I suggest the absence of a quorum., I suggest..."
144,143,101,143_gentlemans_chair_acting_reserve,"[gentlemans, chair, acting, reserve, recognize...",[Madam Chair. I reserve a point of order on th...
145,144,100,144_balance_yield_time_yieldable,"[balance, yield, time, yieldable, closes, sir,...","[I yield back the balance of my time., I yield..."
